# AdaBoost (Adaptive Boosting):

## The idea:

AdaBoost builds a strong classifier from many very weak ones, usually decision **stumps** (trees with a single split). It trains them **one after another**, and each new model concentrates on the samples the previous models got wrong.

Two things make this work:
- **Sample weights:** every training sample has a weight. After each round, misclassified samples get heavier, so the next weak learner is forced to focus on them.
- **Model weights (α):** each weak learner gets a say in the final vote proportional to how accurate it was.

## The algorithm (binary case):

1. Give every sample the same weight, 1/N.
2. For each round m = 1 … M:
   1. Train a weak learner on the **weighted** data.
   2. Compute its weighted error ε, the total weight of the samples it got wrong.
   3. Compute its say: **α = ½ · ln((1 − ε) / ε)**.
   4. Multiply the weights of misclassified samples by e^α and the weights of correct ones by e^−α, then normalise so the weights sum to 1.
3. The final prediction is the sign of the α-weighted vote: **H(x) = sign(Σ αₘ · hₘ(x))**.

How α behaves:

| Weak learner's error ε | α | Meaning |
|---|---|---|
| near 0 | large positive | very reliable, big say |
| 0.5 | 0 | a coin flip, no say |
| above 0.5 | negative | worse than guessing, its vote is flipped |

## A worked example:

Take 10 samples, each with weight 0.1. The first stump misclassifies 3 of them.

- ε = 0.3, so α = ½ · ln(0.7 / 0.3) ≈ **0.424**.
- Misclassified samples are multiplied by e^0.424 ≈ 1.53, and correct ones by e^−0.424 ≈ 0.65.
- After normalising, each misclassified sample has weight **1/6 ≈ 0.167** (up from 0.1), and each correct one has **1/14 ≈ 0.071**.

The 3 misclassified samples now hold a total weight of exactly **0.5**, as do the 7 correct ones. So the first stump is no better than a coin flip on the new weighting, which forces the next stump to learn something different.

## Key hyperparameters:

| Parameter | Effect |
|---|---|
| `estimator` | The weak learner. Keep it weak, since deep trees overfit. |
| `n_estimators` | The number of rounds. More means a more flexible model, up to the point where it starts overfitting. |
| `learning_rate` | Shrinks each model's α. Smaller values need more estimators but usually generalise better. |

## Strengths and weaknesses:

- **Good:** it needs little tuning, works with simple learners, and often beats a single complex model.
- **Bad:** it's **sensitive to noisy labels and outliers**, because it keeps increasing their weights. It also trains sequentially, so unlike bagging it can't be parallelised across trees.

## AdaBoost vs. bagging:

| | Bagging / random forest | AdaBoost |
|---|---|---|
| Training | parallel, independent models | sequential, each depends on the last |
| Data per model | bootstrap sample | the full data with changing weights |
| Base learners | deep, strong trees | shallow, weak trees |
| Vote | equal | weighted by α |
| Mainly reduces | variance | bias |

---